<a href="https://colab.research.google.com/github/galvaf03/Predictive-Aeroacoustics-via-ML/blob/main/aeroacoustics_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predictive Aeroacoustics via ML**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from ipywidgets import interact, FloatSlider, Dropdown
from IPython.display import display, HTML

## **1. Data Loading and Preparation**


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00291/airfoil_self_noise.dat"
columns = ['Frequency_Hz', 'Angle_of_Attack_degrees', 'Chord_Length_m',
           'Free_Stream_Velocity_ms', 'Suction_Side_Displacement_Thickness_m',
           'Scaled_Sound_Pressure_Level_dB']
df = pd.read_csv(url, sep='\t', names=columns)


## **2. Preprocessing**

In [ ]:
# Create a stall target for the classifier
df['is_stall'] = (df['Angle_of_Attack_degrees'] >= 14).astype(int)

X = df.drop(['Scaled_Sound_Pressure_Level_dB', 'is_stall'], axis=1)
y_reg = df['Scaled_Sound_Pressure_Level_dB']
y_class = df['is_stall']

X_train, X_test, y_reg_train, y_reg_test, y_class_train, y_class_test = train_test_split(
    X, y_reg, y_class, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## **3. Model Benchmarking**`

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'K-Neighbors': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'MLP (Neural Net)': MLPRegressor(random_state=42, max_iter=1000),
    'XGBoost': XGBRegressor(random_state=42, eval_metric='rmse'),
    'SVR': SVR(kernel='rbf', C=100),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []


## **4. Training Models and Convergence**

In [ ]:
print("Training models and generating convergence curves...\n")

for name, model in models.items():
    # --- Special Training for XGBoost (to track iterations) ---
    if name == 'XGBoost':
        model.fit(X_train_scaled, y_reg_train,
                  eval_set=[(X_train_scaled, y_reg_train), (X_test_scaled, y_reg_test)],
                  verbose=False)

        # Plot XGBoost convergence
        eval_results = model.evals_result()
        epochs = len(eval_results['validation_0']['rmse'])
        x_axis = range(0, epochs)

        plt.figure(figsize=(8, 4))
        plt.plot(x_axis, eval_results['validation_0']['rmse'], label='Train Error')
        plt.plot(x_axis, eval_results['validation_1']['rmse'], label='Test Error')
        plt.title('Convergence Curve - XGBoost')
        plt.xlabel('Iterations (Trees added)')
        plt.ylabel('Error (RMSE)')
        plt.legend()
        plt.grid(True)
        plt.show()

    # --- Standard Training for other models ---
    else:
        model.fit(X_train_scaled, y_reg_train)

    # --- MLP Convergence Plot ---
    if name == 'MLP (Neural Net)':
        plt.figure(figsize=(8, 4))
        plt.plot(model.loss_curve_, color='blue', linewidth=2)
        plt.title("Convergence Curve - MLP (Neural Network)")
        plt.xlabel("Epochs (Iterations)")
        plt.ylabel("Loss")
        plt.grid(True)
        plt.show()

    # --- Predictions and Metrics ---
    y_pred = model.predict(X_test_scaled)

    r2 = r2_score(y_reg_test, y_pred)
    mae = mean_absolute_error(y_reg_test, y_pred)
    # Mean Absolute Percentage Error (MAPE)
    mape = np.mean(np.abs((y_reg_test - y_pred) / y_reg_test)) * 100

    results.append({'Model': name, 'R2 Score': r2, 'MAE (dB)': mae, 'Relative Error (%)': mape})

# Create results summary and train the classifier
results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
classifier = RandomForestClassifier(random_state=42).fit(X_train_scaled, y_class_train)

print("\n### MODEL EVALUATION ###")
display(results_df)


## **5. Safety and Deployment, 6. Output**

In [ ]:
def predict_and_compare(Frequency_Hz, Angle_of_Attack_degrees, Chord_Length_m,
                        Free_Stream_Velocity_ms, Suction_Side_Displacement_Thickness_m, TimeOfDay):

    # Prepare input
    input_data = pd.DataFrame([[Frequency_Hz, Angle_of_Attack_degrees, Chord_Length_m,
                                Free_Stream_Velocity_ms, Suction_Side_Displacement_Thickness_m]],
                              columns=X.columns)
    input_scaled = scaler.transform(input_data)

    # 1. Get predictions from ALL models for comparison
    preds_all = {name: model.predict(input_scaled)[0] for name, model in models.items()}
    preds_df = pd.DataFrame(list(preds_all.items()), columns=['Model', 'dB']).sort_values(by='dB')

    # 2. Identify best model (highest R2 Score)
    best_model_name = results_df.iloc[0]['Model']
    pred_db_best = preds_all[best_model_name]
    pred_stall = classifier.predict(input_scaled)[0]

    # 3. Alert Logic
    limit = 130 if TimeOfDay == 'Day (7:00 - 23:00)' else 110
    color = "red" if pred_db_best > limit else "green"
    stall_text = "⚠️ STALL DETECTED" if pred_stall == 1 else "✅ SAFE FLIGHT"
    stall_color = "red" if pred_stall == 1 else "green"

    # --- OUTPUT GENERATION ---
    # Clean previous figures to avoid clutter in Colab
    plt.close('all')

    plt.figure(figsize=(10, 4))
    # FIX: Added 'hue' and 'legend=False' to avoid the FutureWarning
    sns.barplot(x='dB', y='Model', data=preds_df, hue='Model', palette='viridis', legend=False)

    plt.axvline(limit, color='red', linestyle='--', label=f'{TimeOfDay} Limit')
    plt.title(f'Real-Time Prediction Comparison (Best Model: {best_model_name})')
    plt.xlim(100, 150)
    plt.legend()
    plt.show()

    display(HTML(f"""
        <div style="border: 2px solid #ddd; padding: 15px; border-radius: 10px; background-color: #f9f9f9; font-family: sans-serif;">
            <h3 style="margin:0;">Analysis Result: <span style="color:{color};">{pred_db_best:.2f} dB</span></h3>
            <p><b>Leading Estimator:</b> {best_model_name}</p>
            <p style="font-size:18px; color:{stall_color}; font-weight:bold;">{stall_text}</p>
            <p style="font-size:12px; color:gray;">Operational Limit for {TimeOfDay}: {limit} dB</p>
        </div>
    """))

# Launching the Interface
interact(predict_and_compare,
         Frequency_Hz=FloatSlider(min=200, max=20000, step=100, value=1000, description='Freq (Hz)'),
         Angle_of_Attack_degrees=FloatSlider(min=0, max=25, step=0.1, value=5.0, description='Angle (°)'),
         Chord_Length_m=FloatSlider(min=0.01, max=0.3, step=0.01, value=0.1, description='Chord (m)'),
         Free_Stream_Velocity_ms=FloatSlider(min=30, max=80, step=1, value=40.0, description='Vel (m/s)'),
         Suction_Side_Displacement_Thickness_m=FloatSlider(min=0, max=0.06, step=0.001, value=0.005, description='Thick (m)'),
         TimeOfDay=Dropdown(options=['Day (7:00 - 23:00)', 'Night (23:00 - 7:00)'], value='Night (23:00 - 7:00)', description='Schedule'));
